Primary Source Context: JAN Workplace Accommodation Data (Job Accommodation Network / U.S. Department of Labor).
> https://github.com/lukeslp/accessibility-atlas-dataset/blob/main/jan_workplace_accommodations.json

Data Provision Status: Synthetic Dataset (Generated via Seeded Script).

In [36]:
import numpy as np
import pandas as pd

# Seeded RNG for reproducibility
np.random.seed(42)
n_samples = 260

# Base Categories
categories = [
    "Schedule modification",
    "Assistive technology",
    "Physical workspace",
    "Policy modification",
    "Communication aid",
]
disability_types = [
    "Physical/Mobility",
    "Sensory",
    "Cognitive/Neurological",
    "Mental Health",
    "Chronic Medical",
]

# Generate base features (Original Code)
disability_col = np.random.choice(disability_types, size=n_samples)
accommodation_col = np.random.choice(
    categories, size=n_samples, p=[0.30, 0.25, 0.20, 0.15, 0.10]
)
tenure_col = np.round(np.random.gamma(shape=2.0, scale=2.5, size=n_samples), 1)
wage_col = np.round(np.random.normal(loc=22.50, scale=5.0, size=n_samples), 2)
degree_col = np.random.choice([1, 0], size=n_samples, p=[0.65, 0.35])

# Binary cost flag (0 = No cost, 1 = Financial expense)
is_cost_col = np.random.choice([0, 1], size=n_samples, p=[0.61, 0.39])

# Calculate accommodation cost based on category relationships
cost_usd = []
for is_cost, acc_type in zip(is_cost_col, accommodation_col):
    if is_cost == 0:
        cost_usd.append(0.0)
    else:
        if acc_type == "Communication aid":  # e.g., Sign language interpreter
            base_cost = np.random.exponential(scale=1800) + 500
        elif acc_type == "Physical workspace":  # e.g., Ramps, automatic doors
            base_cost = np.random.normal(loc=1200, scale=300)
        elif acc_type == "Assistive technology":  # e.g., Screen readers
            base_cost = np.random.normal(loc=450, scale=100)
        else:  # Minor administrative/policy adjustments
            base_cost = np.random.uniform(low=50, high=250)

        cost_usd.append(round(max(10.0, base_cost), 2))

# =====================================================================
# TARGETS FOR BIAS ANALYSIS
# =====================================================================

# 1. Contextual Features
request_severity = np.random.choice(["Low", "Medium", "High"], size=n_samples, p=[0.40, 0.45, 0.15])
remote_status = np.random.choice(["On-site", "Hybrid", "Remote"], size=n_samples, p=[0.50, 0.35, 0.15])
departments = ["Engineering", "Sales", "Customer Support", "Operations", "Finance"]
department_col = np.random.choice(departments, size=n_samples, p=[0.20, 0.25, 0.25, 0.20, 0.10])

# Capped pool of 10 anonymized HR Reviewers
hr_reviewers = [f"HR-10{i}" for i in range(1, 10)] + ["HR-110"]
hr_reviewer_col = np.random.choice(hr_reviewers, size=n_samples)

# 2. Outcomes generated with conditional probabilities (Injecting Synthetic Bias)
approval_status_col = []
denial_reason_col = []
days_to_implement_col = []
retained_employment_col = []

for dis, acc, sev, hr, cost in zip(disability_col, accommodation_col, request_severity, hr_reviewer_col, cost_usd):
    # Baseline probability of approval
    p_approve = 0.75
    
    # Synthetic Disparities: Lower approval for Mental Health/Cognitive, strict reviewer (HR-102), or high cost
    if dis in ["Mental Health", "Cognitive/Neurological"]:
        p_approve -= 0.30
    if hr == "HR-102":
        p_approve -= 0.25
    if cost > 850:
        p_approve -= 0.45
        
    p_approve = max(0.20, min(0.95, p_approve))
    
    status = np.random.choice(
        ["Approved", "Partially Approved", "Denied"],
        p=[p_approve, (1 - p_approve) * 0.4, (1 - p_approve) * 0.6]
    )
    approval_status_col.append(status)
    
    # Denial reason assignment
    if status == "Denied":
        reason = np.random.choice(
            ["Undue Hardship", "Lack of Documentation", "Ineligible", "Alternative Offered"],
            p=[0.35, 0.35, 0.15, 0.15]
        )
        denial_reason_col.append(reason)
        days_to_implement_col.append(np.nan)  # NaN if denied
    else:
        denial_reason_col.append("N/A")
        # Days to implement: Mental Health / Cognitive experience administrative delay
        base_days = 10 if dis in ["Physical/Mobility", "Sensory"] else 22
        days = int(np.round(np.random.gamma(shape=2.0, scale=base_days / 2.0)))
        days_to_implement_col.append(max(1, days))

# 3. Retention outcome based on approval status and fulfillment friction
for status, dis, days in zip(approval_status_col, disability_col, days_to_implement_col):
    p_retain = 0.88
    if status == "Denied":
        p_retain -= 0.35
    if pd.notna(days) and days > 25 and dis in ["Chronic Medical", "Mental Health"]:
        p_retain -= 0.20
    p_retain = max(0.10, min(0.95, p_retain))
    
    retained_employment_col.append(np.random.choice([1, 0], p=[p_retain, 1 - p_retain]))

# Construct full DataFrame
df = pd.DataFrame(
    {
        "request_id": [f"REQ-{1000+i}" for i in range(n_samples)],
        "disability_category": disability_col,
        "accommodation_category": accommodation_col,
        "employee_tenure_years": tenure_col,
        "hourly_wage_usd": wage_col,
        "has_college_degree": degree_col,
        "is_cost_incurred": is_cost_col,
        "accommodation_cost_usd": cost_usd,
        "request_severity_level": request_severity,
        "remote_work_status": remote_status,
        "department": department_col,
        "hr_reviewer_id": hr_reviewer_col,
        "approval_status": approval_status_col,
        "denial_reason": denial_reason_col,
        "days_to_implement": days_to_implement_col,
        "retained_6mo": retained_employment_col,
    }
)

df.to_csv("workplace_accommodation_synthetic.csv", index=False)

In [37]:
dataset = pd.read_csv('workplace_accommodation_synthetic.csv')
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   request_id              260 non-null    str    
 1   disability_category     260 non-null    str    
 2   accommodation_category  260 non-null    str    
 3   employee_tenure_years   260 non-null    float64
 4   hourly_wage_usd         260 non-null    float64
 5   has_college_degree      260 non-null    int64  
 6   is_cost_incurred        260 non-null    int64  
 7   accommodation_cost_usd  260 non-null    float64
 8   request_severity_level  260 non-null    str    
 9   remote_work_status      260 non-null    str    
 10  department              260 non-null    str    
 11  hr_reviewer_id          260 non-null    str    
 12  approval_status         260 non-null    str    
 13  denial_reason           69 non-null     str    
 14  days_to_implement       191 non-null    float64
 15  

In [38]:
dataset.describe()

,employee_tenure_years,hourly_wage_usd,has_college_degree,is_cost_incurred,accommodation_cost_usd,days_to_implement,retained_6mo
count,260.000000,260.000000,260.000000,260.000000,260.000000,191.000000,260.000000
mean,5.460769,23.103385,0.684615,0.380769,260.160769,18.382199,0.784615
std,3.988177,5.340462,0.465565,0.486512,645.648811,13.469478,0.411882
min,0.400000,9.450000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,2.400000,19.562500,0.000000,0.000000,0.000000,9.000000,1.000000
50%,4.500000,22.630000,1.000000,0.000000,0.000000,15.000000,1.000000
75%,7.600000,26.662500,1.000000,1.000000,210.995000,25.000000,1.000000
max,20.800000,38.340000,1.000000,1.000000,7588.770000,72.000000,1.000000


In [39]:
dataset.head(5)

,request_id,disability_category,accommodation_category,employee_tenure_years,hourly_wage_usd,has_college_degree,is_cost_incurred,accommodation_cost_usd,request_severity_level,remote_work_status,department,hr_reviewer_id,approval_status,denial_reason,days_to_implement,retained_6mo
0,REQ-1000,Mental Health,Assistive technology,4.3,22.46,1,0,0.00,Medium,Hybrid,Sales,HR-107,Denied,Undue Hardship,NaN,0
1,REQ-1001,Chronic Medical,Assistive technology,5.6,21.83,1,0,0.00,Low,Remote,Operations,HR-101,Denied,Alternative Offered,NaN,0
2,REQ-1002,Cognitive/Neurological,Communication aid,7.5,26.43,1,1,2321.51,Medium,Hybrid,Engineering,HR-103,Approved,NaN,16.0,1
3,REQ-1003,Chronic Medical,Schedule modification,1.6,20.24,1,0,0.00,Low,On-site,Operations,HR-104,Denied,Undue Hardship,NaN,0
4,REQ-1004,Chronic Medical,Schedule modification,2.3,25.84,0,1,79.80,Low,Hybrid,Operations,HR-104,Approved,NaN,19.0,1


In [46]:
print("Unique `disability_category`:", dataset['disability_category'].unique().tolist())

print("\nUnique `accommodation_category`:", dataset['accommodation_category'].unique().tolist())

print("\nUnique `remote_work_status`:", dataset['remote_work_status'].unique().tolist())

print("\nUnique `department`:", dataset['department'].unique().tolist())

print("\nUnique `hr_reviewer_id`:", dataset['hr_reviewer_id'].unique().tolist())

print("\nUnique `denial_reason`:", dataset['denial_reason'].unique().tolist())

print("\nUnique `request_severity_level`:", dataset['request_severity_level'].unique().tolist())

Unique `disability_category`: ['Mental Health', 'Chronic Medical', 'Cognitive/Neurological', 'Sensory', 'Physical/Mobility']

Unique `accommodation_category`: ['Assistive technology', 'Communication aid', 'Schedule modification', 'Physical workspace', 'Policy modification']

Unique `remote_work_status`: ['Hybrid', 'Remote', 'On-site']

Unique `department`: ['Sales', 'Operations', 'Engineering', 'Customer Support', 'Finance']

Unique `hr_reviewer_id`: ['HR-107', 'HR-101', 'HR-103', 'HR-104', 'HR-105', 'HR-110', 'HR-102', 'HR-109', 'HR-106', 'HR-108']

Unique `denial_reason`: ['Undue Hardship', 'Alternative Offered', nan, 'Lack of Documentation', 'Ineligible']

Unique `request_severity_level`: ['Medium', 'Low', 'High']
